In [1]:
# Clone the repo
!git clone https://github.com/nhatvu205/innovations-on-graph-wavenet-for-spatial-temporal-graph-modeling.git
%cd innovations-on-graph-wavenet-for-spatial-temporal-graph-modeling

Cloning into 'innovations-on-graph-wavenet-for-spatial-temporal-graph-modeling'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 88 (delta 29), reused 72 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 3.86 MiB | 35.65 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/kaggle/working/innovations-on-graph-wavenet-for-spatial-temporal-graph-modeling


In [2]:
# Install dependencies
!pip install -r requirements.txt

In [3]:
import sys
sys.path.insert(0, '/kaggle/working/innovations-on-graph-wavenet-for-spatial-temporal-graph-modeling')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import time

from src import model, util, engine as eng

# ─── Config ─────────────────────────────────────────────────────
DEVICE      = torch.device("cuda:0")
DATA        = '/kaggle/input/datasets/nhl875/pems-bay/PEMS-BAY/PEMS-BAY'
ADJDATA     = '/kaggle/input/datasets/izmildqnh/metr-la/sensor_graph/sensor_graph/adj_mx_bay.pkl'
ADJTYPE     = 'doubletransition'
NUM_NODES   = 325
IN_DIM      = 1
SEQ_LENGTH  = 12
NHID        = 32
DROPOUT     = 0.3
BATCH_SIZE  = 64
LR          = 0.001
WDECAY      = 0.0001
EPOCHS      = 100
PRINT_EVERY = 50
SAVE        = '/kaggle/working/gwnet_dynamic'
EXPID       = 1
DYNAMIC_ADJ = True

# ─── Class definitions ──────────────────────────────────────────
class DynamicAdaptiveAdj(nn.Module):
    def __init__(self, num_nodes, emb_dim=10, in_channels=32):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, emb_dim, kernel_size=(1, 1), bias=True)

    def forward(self, x, nodevec1, nodevec2):
        z = x.mean(dim=-1, keepdim=True)
        z = self.proj(z).squeeze(-1)
        z = z.permute(0, 2, 1)
        nv1 = nodevec1.unsqueeze(0) + z
        nv2 = nodevec2.t().unsqueeze(0) + z
        logits = torch.bmm(nv1, nv2.permute(0, 2, 1))
        return F.softmax(F.relu(logits), dim=-1)


class gcn_patched(nn.Module):
    def __init__(self, c_in, c_out, dropout, support_len=3, order=2):
        super().__init__()
        self.nconv = model.nconv()
        c_in = (order * support_len + 1) * c_in
        self.mlp = model.linear(c_in, c_out)
        self.dropout = dropout
        self.order = order

    def _nconv_dynamic(self, x, A):
        return torch.einsum('bcnl,bnm->bcml', x, A).contiguous()

    def forward(self, x, support):
        out = [x]
        for a in support:
            conv_fn = self._nconv_dynamic if a.dim() == 3 else self.nconv
            x1 = conv_fn(x, a)
            out.append(x1)
            for k in range(2, self.order + 1):
                x2 = conv_fn(x1, a)
                out.append(x2)
                x1 = x2
        h = torch.cat(out, dim=1)
        h = self.mlp(h)
        h = F.dropout(h, self.dropout, training=self.training)
        return h


# ─── Patch gwnet ────────────────────────────────────────────────
_original_gwnet_init = model.gwnet.__init__

def _patched_gwnet_init(self, *args, dynamic_adj=False, **kwargs):
    _original_gwnet_init(self, *args, **kwargs)
    self.dynamic_adj = dynamic_adj and self.addaptadj
    if self.dynamic_adj:
        dilation_channels = self.filter_convs[0].out_channels
        residual_channels = self.start_conv.out_channels
        num_nodes         = self.nodevec1.shape[0]
        self.dyn_adj = DynamicAdaptiveAdj(
            num_nodes=num_nodes, emb_dim=10, in_channels=dilation_channels
        )
        support_len = len(self.supports) + 1
        new_gconv = nn.ModuleList()
        for g in self.gconv:
            new_gconv.append(gcn_patched(
                c_in=residual_channels,
                c_out=g.mlp.mlp.out_channels,
                dropout=g.dropout,
                support_len=support_len,
                order=g.order,
            ))
        self.gconv = new_gconv


def _patched_gwnet_forward(self, input):
    in_len = input.size(3)
    if in_len < self.receptive_field:
        x = F.pad(input, (self.receptive_field - in_len, 0, 0, 0))
    else:
        x = input
    x = self.start_conv(x)
    skip = 0
    static_supports = self.supports if self.supports is not None else []
    if self.gcn_bool and self.supports is not None and self.addaptadj and not self.dynamic_adj:
        adp = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
        new_supports = static_supports + [adp]
    for i in range(self.blocks * self.layers):
        residual = x
        filter_ = torch.tanh(self.filter_convs[i](residual))
        gate    = torch.sigmoid(self.gate_convs[i](residual))
        x = filter_ * gate
        s = self.skip_convs[i](x)
        if not isinstance(skip, int):
            skip = skip[:, :, :, -s.size(3):]
        skip = s + skip
        if self.gcn_bool and self.supports is not None:
            if self.addaptadj and self.dynamic_adj:
                adp_dyn          = self.dyn_adj(x, self.nodevec1, self.nodevec2)
                current_supports = static_supports + [adp_dyn]
                x = self.gconv[i](x, current_supports)
            elif self.addaptadj:
                x = self.gconv[i](x, new_supports)
            else:
                x = self.gconv[i](x, self.supports)
        else:
            x = self.residual_convs[i](x)
        x = x + residual[:, :, :, -x.size(3):]
        x = self.bn[i](x)
    x = F.relu(skip)
    x = F.relu(self.end_conv_1(x))
    x = self.end_conv_2(x)
    return x


def _patched_trainer_init(self, scaler, in_dim, seq_length, num_nodes,
                           nhid, dropout, lrate, wdecay, device,
                           supports, gcn_bool, addaptadj, aptinit,
                           dynamic_adj=False):
    self.model = model.gwnet(
        device, num_nodes, dropout,
        supports=supports, gcn_bool=gcn_bool,
        addaptadj=addaptadj, aptinit=aptinit,
        in_dim=in_dim, out_dim=seq_length,
        residual_channels=nhid, dilation_channels=nhid,
        skip_channels=nhid * 8, end_channels=nhid * 16,
        dynamic_adj=dynamic_adj,
    )
    self.model.to(device)
    self.optimizer = optim.Adam(self.model.parameters(), lr=lrate, weight_decay=wdecay)
    self.loss   = util.masked_mae
    self.scaler = scaler
    self.clip   = 5

model.gwnet.__init__   = _patched_gwnet_init
model.gwnet.forward    = _patched_gwnet_forward
eng.trainer.__init__   = _patched_trainer_init
model.DynamicAdaptiveAdj = DynamicAdaptiveAdj
model.gcn_patched        = gcn_patched
print("Patches applied.")

# ─── Data ───────────────────────────────────────────────────────
sensor_ids, sensor_id_to_ind, adj_mx = util.load_adj(ADJDATA, ADJTYPE)
dataloader = util.load_dataset(DATA, BATCH_SIZE, BATCH_SIZE, BATCH_SIZE)
scaler     = dataloader["scaler"]
supports   = [torch.tensor(i).to(DEVICE) for i in adj_mx]
adjinit    = supports[0]

# ─── Engine ─────────────────────────────────────────────────────
engine = eng.trainer(
    scaler, IN_DIM, SEQ_LENGTH, NUM_NODES,
    NHID, DROPOUT, LR, WDECAY, DEVICE,
    supports, gcn_bool=True, addaptadj=True,
    aptinit=adjinit, dynamic_adj=DYNAMIC_ADJ,
)

# Verify
total_p = sum(p.numel() for p in engine.model.parameters())
opt_p   = sum(p.numel() for pg in engine.optimizer.param_groups for p in pg['params'])
dyn_p   = sum(p.numel() for p in engine.model.dyn_adj.parameters())
print(f"Total params    : {total_p:,}")
print(f"dyn_adj params  : {dyn_p:,}")
print(f"Optimizer params: {opt_p:,}")
assert total_p == opt_p, "Optimizer thiếu params!"
print("Param check passed.\n")

# ─── Training ───────────────────────────────────────────────────
t_total_start = time.time()
his_loss, val_time, train_time = [], [], []
best_val_loss = float('inf')          # ← THÊM
best_epoch    = 1                     # ← THÊM

for epoch in range(1, EPOCHS + 1):
    train_loss, train_mape, train_rmse = [], [], []
    t1 = time.time()
    dataloader["train_loader"].shuffle()

    for i, (x, y) in enumerate(dataloader["train_loader"].get_iterator()):
        trainx = torch.Tensor(x).to(DEVICE).transpose(1, 3)
        trainy = torch.Tensor(y).to(DEVICE).transpose(1, 3)
        metrics = engine.train(trainx, trainy[:, 0, :, :])
        train_loss.append(metrics[0])
        train_mape.append(metrics[1])
        train_rmse.append(metrics[2])
        if i % PRINT_EVERY == 0:
            print(f"Iter: {i:03d}, Train Loss: {train_loss[-1]:.4f}, "
                  f"Train MAPE: {train_mape[-1]:.4f}, Train RMSE: {train_rmse[-1]:.4f}",
                  flush=True)
    t2 = time.time()
    train_time.append(t2 - t1)

    valid_loss, valid_mape, valid_rmse = [], [], []
    s1 = time.time()
    for x, y in dataloader["val_loader"].get_iterator():
        valx = torch.Tensor(x).to(DEVICE).transpose(1, 3)
        valy = torch.Tensor(y).to(DEVICE).transpose(1, 3)
        metrics = engine.eval(valx, valy[:, 0, :, :])
        valid_loss.append(metrics[0])
        valid_mape.append(metrics[1])
        valid_rmse.append(metrics[2])
    s2 = time.time()
    val_time.append(s2 - s1)

    mtrain_loss = np.mean(train_loss)
    mvalid_loss = np.mean(valid_loss)
    his_loss.append(mvalid_loss)

    print(f"Epoch: {epoch:03d}, Inference Time: {s2 - s1:.4f} secs")
    print(f"Epoch: {epoch:03d}, "
          f"Train Loss: {mtrain_loss:.4f}, Train MAPE: {np.mean(train_mape):.4f}, "
          f"Train RMSE: {np.mean(train_rmse):.4f}, Valid Loss: {mvalid_loss:.4f}, "
          f"Valid MAPE: {np.mean(valid_mape):.4f}, Valid RMSE: {np.mean(valid_rmse):.4f}, "
          f"Training Time: {t2 - t1:.4f}/epoch", flush=True)

    # ── Lưu checkpoint mỗi epoch ────────────────────────────────
    ckpt_path = f"{SAVE}_epoch_{epoch}_{round(mvalid_loss, 2):.2f}.pth"
    torch.save(engine.model.state_dict(), ckpt_path)

    # ── Lưu riêng best checkpoint ngay khi tìm thấy ─────────────
    if mvalid_loss < best_val_loss:
        best_val_loss = mvalid_loss
        best_epoch    = epoch
        torch.save(
            engine.model.state_dict(),
            f"{SAVE}_best.pth",           # ← tên cố định, luôn là best
        )
        print(f"  → New best saved (epoch {epoch}, val_loss={best_val_loss:.4f})",
              flush=True)
        
# ─── Test ───────────────────────────────────────────────────────
print(f"\nAverage Training Time: {np.mean(train_time):.4f} secs/epoch")
print(f"Average Inference Time: {np.mean(val_time):.4f} secs")

# Load best — dùng file cố định thay vì reconstruct tên từ his_loss
best_path = f"{SAVE}_best.pth"
engine.model.load_state_dict(torch.load(best_path))
print(f"Training finished. Best validation loss: {best_val_loss:.4f} (epoch {best_epoch})")

# ── tính metric từng batch, không cat toàn bộ → tránh OOM ──────
amae  = [[] for _ in range(SEQ_LENGTH)]
amape = [[] for _ in range(SEQ_LENGTH)]
armse = [[] for _ in range(SEQ_LENGTH)]

engine.model.eval()
for x, y in dataloader["test_loader"].get_iterator():
    testx = torch.Tensor(x).to(DEVICE).transpose(1, 3)
    testy = torch.Tensor(y).to(DEVICE).transpose(1, 3)[:, 0, :, :]   # (B, N, T)
    with torch.no_grad():
        preds = engine.model(
            nn.functional.pad(testx, (1, 0, 0, 0))
        ).transpose(1, 3).squeeze(1)                                    # (B, N, T)
    for i in range(SEQ_LENGTH):
        pred = scaler.inverse_transform(preds[:, :, i])
        real = testy[:, :, i]
        mae, mape, rmse = util.metric(pred, real)
        amae[i].append(mae)
        amape[i].append(mape)
        armse[i].append(rmse)

avg_mae, avg_mape, avg_rmse = [], [], []
for i in range(SEQ_LENGTH):
    m, p, r = np.mean(amae[i]), np.mean(amape[i]), np.mean(armse[i])
    avg_mae.append(m); avg_mape.append(p); avg_rmse.append(r)
    print(f"Evaluate best model on test data for horizon {i+1:02d}, "
          f"Test MAE: {m:.4f}, Test MAPE: {p:.4f}, Test RMSE: {r:.4f}")

print(f"\nOn average over {SEQ_LENGTH} horizons, "
      f"Test MAE: {np.mean(avg_mae):.4f}, "
      f"Test MAPE: {np.mean(avg_mape):.4f}, "
      f"Test RMSE: {np.mean(avg_rmse):.4f}")

torch.save(engine.model.state_dict(),
           f"{SAVE}_exp{EXPID}_best_{round(his_loss[bestid], 2):.2f}.pth")

print(f"Total time spent: {time.time() - t_total_start:.4f}")

Patches applied.
Total params    : 312,058
dyn_adj params  : 330
Optimizer params: 312,058
Param check passed.

Iter: 000, Train Loss: 5.3196, Train MAPE: 0.1289, Train RMSE: 8.8720
Iter: 050, Train Loss: 2.2054, Train MAPE: 0.0479, Train RMSE: 5.1708
Iter: 100, Train Loss: 2.1168, Train MAPE: 0.0512, Train RMSE: 4.7030
Iter: 150, Train Loss: 2.2782, Train MAPE: 0.0551, Train RMSE: 4.9690
Iter: 200, Train Loss: 1.9287, Train MAPE: 0.0466, Train RMSE: 4.5148
Iter: 250, Train Loss: 2.2747, Train MAPE: 0.0551, Train RMSE: 4.9028
Iter: 300, Train Loss: 2.0270, Train MAPE: 0.0534, Train RMSE: 4.4737
Iter: 350, Train Loss: 2.1702, Train MAPE: 0.0530, Train RMSE: 4.9577
Iter: 400, Train Loss: 1.9102, Train MAPE: 0.0434, Train RMSE: 4.2326
Iter: 450, Train Loss: 1.7737, Train MAPE: 0.0359, Train RMSE: 3.9862
Iter: 500, Train Loss: 1.6804, Train MAPE: 0.0366, Train RMSE: 3.8513
Iter: 550, Train Loss: 1.8586, Train MAPE: 0.0431, Train RMSE: 4.0810
Epoch: 001, Inference Time: 11.2344 secs
Epoch: 

NameError: name 'bestid' is not defined

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/innovations-on-graph-wavenet-for-spatial-temporal-graph-modeling')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import time

from src import model, util, engine as eng

# ─── Config ─────────────────────────────────────────────────────
DEVICE      = torch.device("cuda:0")
DATA        = '/kaggle/input/datasets/nhl875/pems-bay/PEMS-BAY/PEMS-BAY'
ADJDATA     = '/kaggle/input/datasets/izmildqnh/metr-la/sensor_graph/sensor_graph/adj_mx_bay.pkl'
ADJTYPE     = 'doubletransition'
NUM_NODES   = 325
IN_DIM      = 1
SEQ_LENGTH  = 12
NHID        = 32
DROPOUT     = 0.3
BATCH_SIZE  = 64
LR          = 0.001
WDECAY      = 0.0001
EPOCHS      = 100
PRINT_EVERY = 50
SAVE        = '/kaggle/working/gwnet_dynamic'
EXPID       = 1
DYNAMIC_ADJ = True

# ─── Class definitions ──────────────────────────────────────────
class DynamicAdaptiveAdj(nn.Module):
    def __init__(self, num_nodes, emb_dim=10, in_channels=32):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, emb_dim, kernel_size=(1, 1), bias=True)

    def forward(self, x, nodevec1, nodevec2):
        z = x.mean(dim=-1, keepdim=True)
        z = self.proj(z).squeeze(-1)
        z = z.permute(0, 2, 1)
        nv1 = nodevec1.unsqueeze(0) + z
        nv2 = nodevec2.t().unsqueeze(0) + z
        logits = torch.bmm(nv1, nv2.permute(0, 2, 1))
        return F.softmax(F.relu(logits), dim=-1)


class gcn_patched(nn.Module):
    def __init__(self, c_in, c_out, dropout, support_len=3, order=2):
        super().__init__()
        self.nconv = model.nconv()
        c_in = (order * support_len + 1) * c_in
        self.mlp = model.linear(c_in, c_out)
        self.dropout = dropout
        self.order = order

    def _nconv_dynamic(self, x, A):
        return torch.einsum('bcnl,bnm->bcml', x, A).contiguous()

    def forward(self, x, support):
        out = [x]
        for a in support:
            conv_fn = self._nconv_dynamic if a.dim() == 3 else self.nconv
            x1 = conv_fn(x, a)
            out.append(x1)
            for k in range(2, self.order + 1):
                x2 = conv_fn(x1, a)
                out.append(x2)
                x1 = x2
        h = torch.cat(out, dim=1)
        h = self.mlp(h)
        h = F.dropout(h, self.dropout, training=self.training)
        return h


# ─── Patch gwnet ────────────────────────────────────────────────
_original_gwnet_init = model.gwnet.__init__

def _patched_gwnet_init(self, *args, dynamic_adj=False, **kwargs):
    _original_gwnet_init(self, *args, **kwargs)
    self.dynamic_adj = dynamic_adj and self.addaptadj
    if self.dynamic_adj:
        dilation_channels = self.filter_convs[0].out_channels
        residual_channels = self.start_conv.out_channels
        num_nodes         = self.nodevec1.shape[0]
        self.dyn_adj = DynamicAdaptiveAdj(
            num_nodes=num_nodes, emb_dim=10, in_channels=dilation_channels
        )
        support_len = len(self.supports) + 1
        new_gconv = nn.ModuleList()
        for g in self.gconv:
            new_gconv.append(gcn_patched(
                c_in=residual_channels,
                c_out=g.mlp.mlp.out_channels,
                dropout=g.dropout,
                support_len=support_len,
                order=g.order,
            ))
        self.gconv = new_gconv


def _patched_gwnet_forward(self, input):
    in_len = input.size(3)
    if in_len < self.receptive_field:
        x = F.pad(input, (self.receptive_field - in_len, 0, 0, 0))
    else:
        x = input
    x = self.start_conv(x)
    skip = 0
    static_supports = self.supports if self.supports is not None else []
    if self.gcn_bool and self.supports is not None and self.addaptadj and not self.dynamic_adj:
        adp = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
        new_supports = static_supports + [adp]
    for i in range(self.blocks * self.layers):
        residual = x
        filter_ = torch.tanh(self.filter_convs[i](residual))
        gate    = torch.sigmoid(self.gate_convs[i](residual))
        x = filter_ * gate
        s = self.skip_convs[i](x)
        if not isinstance(skip, int):
            skip = skip[:, :, :, -s.size(3):]
        skip = s + skip
        if self.gcn_bool and self.supports is not None:
            if self.addaptadj and self.dynamic_adj:
                adp_dyn          = self.dyn_adj(x, self.nodevec1, self.nodevec2)
                current_supports = static_supports + [adp_dyn]
                x = self.gconv[i](x, current_supports)
            elif self.addaptadj:
                x = self.gconv[i](x, new_supports)
            else:
                x = self.gconv[i](x, self.supports)
        else:
            x = self.residual_convs[i](x)
        x = x + residual[:, :, :, -x.size(3):]
        x = self.bn[i](x)
    x = F.relu(skip)
    x = F.relu(self.end_conv_1(x))
    x = self.end_conv_2(x)
    return x


def _patched_trainer_init(self, scaler, in_dim, seq_length, num_nodes,
                           nhid, dropout, lrate, wdecay, device,
                           supports, gcn_bool, addaptadj, aptinit,
                           dynamic_adj=False):
    self.model = model.gwnet(
        device, num_nodes, dropout,
        supports=supports, gcn_bool=gcn_bool,
        addaptadj=addaptadj, aptinit=aptinit,
        in_dim=in_dim, out_dim=seq_length,
        residual_channels=nhid, dilation_channels=nhid,
        skip_channels=nhid * 8, end_channels=nhid * 16,
        dynamic_adj=dynamic_adj,
    )
    self.model.to(device)
    self.optimizer = optim.Adam(self.model.parameters(), lr=lrate, weight_decay=wdecay)
    self.loss   = util.masked_mae
    self.scaler = scaler
    self.clip   = 5

model.gwnet.__init__   = _patched_gwnet_init
model.gwnet.forward    = _patched_gwnet_forward
eng.trainer.__init__   = _patched_trainer_init
model.DynamicAdaptiveAdj = DynamicAdaptiveAdj
model.gcn_patched        = gcn_patched
print("Patches applied.")

# ─── Data ───────────────────────────────────────────────────────
sensor_ids, sensor_id_to_ind, adj_mx = util.load_adj(ADJDATA, ADJTYPE)
dataloader = util.load_dataset(DATA, BATCH_SIZE, BATCH_SIZE, BATCH_SIZE)
scaler     = dataloader["scaler"]
supports   = [torch.tensor(i).to(DEVICE) for i in adj_mx]
adjinit    = supports[0]

# ─── Engine ─────────────────────────────────────────────────────
engine = eng.trainer(
    scaler, IN_DIM, SEQ_LENGTH, NUM_NODES,
    NHID, DROPOUT, LR, WDECAY, DEVICE,
    supports, gcn_bool=True, addaptadj=True,
    aptinit=adjinit, dynamic_adj=DYNAMIC_ADJ,
)

# Verify
total_p = sum(p.numel() for p in engine.model.parameters())
opt_p   = sum(p.numel() for pg in engine.optimizer.param_groups for p in pg['params'])
dyn_p   = sum(p.numel() for p in engine.model.dyn_adj.parameters())
print(f"Total params    : {total_p:,}")
print(f"dyn_adj params  : {dyn_p:,}")
print(f"Optimizer params: {opt_p:,}")
assert total_p == opt_p, "Optimizer thiếu params!"
print("Param check passed.\n")

# ─── Training ───────────────────────────────────────────────────
t_total_start = time.time()
his_loss, val_time, train_time = [], [], []
best_val_loss = float('inf')          # ← THÊM
best_epoch    = 1                     # ← THÊM

for epoch in range(1, EPOCHS + 1):
    train_loss, train_mape, train_rmse = [], [], []
    t1 = time.time()
    dataloader["train_loader"].shuffle()

    for i, (x, y) in enumerate(dataloader["train_loader"].get_iterator()):
        trainx = torch.Tensor(x).to(DEVICE).transpose(1, 3)
        trainy = torch.Tensor(y).to(DEVICE).transpose(1, 3)
        metrics = engine.train(trainx, trainy[:, 0, :, :])
        train_loss.append(metrics[0])
        train_mape.append(metrics[1])
        train_rmse.append(metrics[2])
        if i % PRINT_EVERY == 0:
            print(f"Iter: {i:03d}, Train Loss: {train_loss[-1]:.4f}, "
                  f"Train MAPE: {train_mape[-1]:.4f}, Train RMSE: {train_rmse[-1]:.4f}",
                  flush=True)
    t2 = time.time()
    train_time.append(t2 - t1)

    valid_loss, valid_mape, valid_rmse = [], [], []
    s1 = time.time()
    for x, y in dataloader["val_loader"].get_iterator():
        valx = torch.Tensor(x).to(DEVICE).transpose(1, 3)
        valy = torch.Tensor(y).to(DEVICE).transpose(1, 3)
        metrics = engine.eval(valx, valy[:, 0, :, :])
        valid_loss.append(metrics[0])
        valid_mape.append(metrics[1])
        valid_rmse.append(metrics[2])
    s2 = time.time()
    val_time.append(s2 - s1)

    mtrain_loss = np.mean(train_loss)
    mvalid_loss = np.mean(valid_loss)
    his_loss.append(mvalid_loss)

    print(f"Epoch: {epoch:03d}, Inference Time: {s2 - s1:.4f} secs")
    print(f"Epoch: {epoch:03d}, "
          f"Train Loss: {mtrain_loss:.4f}, Train MAPE: {np.mean(train_mape):.4f}, "
          f"Train RMSE: {np.mean(train_rmse):.4f}, Valid Loss: {mvalid_loss:.4f}, "
          f"Valid MAPE: {np.mean(valid_mape):.4f}, Valid RMSE: {np.mean(valid_rmse):.4f}, "
          f"Training Time: {t2 - t1:.4f}/epoch", flush=True)

    # ── Lưu checkpoint mỗi epoch ────────────────────────────────
    ckpt_path = f"{SAVE}_epoch_{epoch}_{round(mvalid_loss, 2):.2f}.pth"
    torch.save(engine.model.state_dict(), ckpt_path)

    # ── Lưu riêng best checkpoint ngay khi tìm thấy ─────────────
    if mvalid_loss < best_val_loss:
        best_val_loss = mvalid_loss
        best_epoch    = epoch
        torch.save(
            engine.model.state_dict(),
            f"{SAVE}_best.pth",           # ← tên cố định, luôn là best
        )
        print(f"  → New best saved (epoch {epoch}, val_loss={best_val_loss:.4f})",
              flush=True)

# ─── Sau vòng training, lưu thêm file tổng kết ─────────────────
print(f"\nAverage Training Time: {np.mean(train_time):.4f} secs/epoch")
print(f"Average Inference Time: {np.mean(val_time):.4f} secs")

# Load best
best_path = f"{SAVE}_best.pth"
engine.model.load_state_dict(torch.load(best_path))
print(f"Training finished. Best validation loss: {best_val_loss:.4f} (epoch {best_epoch})")

# ── Test metrics ────────────────────────────────────────────────
amae  = [[] for _ in range(SEQ_LENGTH)]
amape = [[] for _ in range(SEQ_LENGTH)]
armse = [[] for _ in range(SEQ_LENGTH)]

engine.model.eval()
for x, y in dataloader["test_loader"].get_iterator():
    testx = torch.Tensor(x).to(DEVICE).transpose(1, 3)
    testy = torch.Tensor(y).to(DEVICE).transpose(1, 3)[:, 0, :, :]
    with torch.no_grad():
        preds = engine.model(
            nn.functional.pad(testx, (1, 0, 0, 0))
        ).transpose(1, 3).squeeze(1)
    for i in range(SEQ_LENGTH):
        pred = scaler.inverse_transform(preds[:, :, i])
        real = testy[:, :, i]
        mae, mape, rmse = util.metric(pred, real)
        amae[i].append(mae)
        amape[i].append(mape)
        armse[i].append(rmse)

avg_mae, avg_mape, avg_rmse = [], [], []
for i in range(SEQ_LENGTH):
    m, p, r = np.mean(amae[i]), np.mean(amape[i]), np.mean(armse[i])
    avg_mae.append(m); avg_mape.append(p); avg_rmse.append(r)
    print(f"Evaluate best model on test data for horizon {i+1:02d}, "
          f"Test MAE: {m:.4f}, Test MAPE: {p:.4f}, Test RMSE: {r:.4f}")

print(f"\nOn average over {SEQ_LENGTH} horizons, "
      f"Test MAE: {np.mean(avg_mae):.4f}, "
      f"Test MAPE: {np.mean(avg_mape):.4f}, "
      f"Test RMSE: {np.mean(avg_rmse):.4f}")

# ── Lưu file cuối với tên rõ ràng (sửa lỗi bestid) ──────────────
torch.save(
    engine.model.state_dict(),
    f"{SAVE}_exp{EXPID}_best_epoch{best_epoch}_{best_val_loss:.2f}.pth"
)

print(f"Total time spent: {time.time() - t_total_start:.4f}")